# Column Vecchia real-data resolution sensitivity: every-k grid thinning

This notebook fits only the batched reverse-L Column Vecchia model while increasing the data density.

Unlike the earlier `real_data_resolution_*` notebooks, this does **not** take the first `N/R` rows from max-min ordering.  Instead, it constructs a deterministic regular-grid order and keeps `0, R, 2R, ...` for `R = 8, 4, 2, 1`.  Thus `x8` is sparse, `x4` is denser, `x2` is denser again, and `x1` uses the full grid.

Model fixed here:

- Column ST, reverse-L geometry: `Up2_Right3_Down14_Lag2`, `head_right_cols=0`
- Covariance coordinates: real `Source_Latitude` / `Source_Longitude`
- Scan/order coordinates: regular `Latitude` / `Longitude` grid
- Mean design: `hour_spatial`

In [ ]:
import gc
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

GEMS_ROOT = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
SRC_ROOT = GEMS_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_st_trend_050726 import ColumnSTTrendVecchiaFit

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

In [ ]:
# Experiment controls
DEVICE = torch.device('cpu')
DTYPE = torch.float64

YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [2]          # 0-based: 2 -> 2024-07-03. Add more days if needed.
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

RESOLUTION_STRIDES = [8, 4, 2, 1]
FIT_SMOOTHS = [0.5]
MEAN_DESIGN = 'hour_spatial'

LBFGS_LR = 1.0
LBFGS_STEPS = 5
LBFGS_EVAL = 15
LBFGS_HIST = 10
GRAD_TOL = 1e-5

COLUMN_SPEC = {
    'model': 'ColumnResolutionStride_Up2_Right3_Down14_Lag2_head0_realloc',
    'head_right_cols': 0,
    'above_count': 2,
    'right_col_count': 3,
    'per_lag_conditioning_count': 14,
    'lag_count': 2,
    'include_lag_self': False,
    'target_chunk_size': 512,
}

INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

OUT_DIR = GEMS_ROOT / 'Exercises/st_model/day/local_computer/testing/log'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = 'real_column_resolution_stride_050726'

print('Column spec:', COLUMN_SPEC)
print('strides:', RESOLUTION_STRIDES)
print('mean_design:', MEAN_DESIGN)

In [ ]:
def init_to_raw(d):
    phi2 = 1.0 / d['range_lon']
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    phi1 = d['sigmasq'] * phi2
    return [
        np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
        d['advec_lat'], d['advec_lon'], np.log(d['nugget']),
    ]

INIT_LOG = init_to_raw(INIT_DICT)


def make_params():
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in INIT_LOG]


def backmap_params(out):
    vals = [float(x) for x in out[:-1]]
    phi1 = np.exp(vals[0])
    phi2 = np.exp(vals[1])
    phi3 = np.exp(vals[2])
    phi4 = np.exp(vals[3])
    return {
        'sigmasq': phi1 / phi2,
        'range_lon': 1.0 / phi2,
        'range_lat': 1.0 / (phi2 * np.sqrt(phi3)),
        'range_time': 1.0 / (phi2 * np.sqrt(phi4)),
        'advec_lat': vals[4],
        'advec_lon': vals[5],
        'nugget': np.exp(vals[6]),
    }


def count_valid(input_map):
    return int(sum((~torch.isnan(v[:, 2])).sum().item() for v in input_map.values()))


def column_diag(model):
    groups = getattr(model, 'Batched_Groups', []) or []
    if not groups:
        return {
            'n_batches': 0, 'largest_batch_n': np.nan, 'mean_m': np.nan,
            'max_m': np.nan, 'n_tails': getattr(model, 'n_tails', np.nan),
        }

    # ReverseLColumnVecchiaFitV3 stores batch dictionaries with
    # target_idx/max_m.  Keep fallbacks for older experimental kernels.
    ns = []
    ms = []
    for g in groups:
        if 'target_idx' in g:
            ns.append(int(g['target_idx'].shape[0]))
        elif 'targets' in g:
            ns.append(int(g['targets'].numel()))
        elif 'idx' in g:
            ns.append(int(g['idx'].numel()))

        if 'max_m' in g:
            ms.append(int(g['max_m']))
        elif 'm' in g:
            ms.append(int(g['m']))
        elif 'neigh_idx' in g:
            ms.append(int(g['neigh_idx'].shape[1]))

    ns = np.asarray(ns, dtype=float)
    ms = np.asarray(ms, dtype=float)
    return {
        'n_batches': len(groups),
        'largest_batch_n': float(ns.max()) if ns.size else np.nan,
        'mean_m': float(ms.mean()) if ms.size else np.nan,
        'max_m': int(ms.max()) if ms.size else np.nan,
        'n_tails': int(ns.sum()) if ns.size else getattr(model, 'n_tails', np.nan),
    }


def round_float_columns(df, ndigits=4):
    out = df.copy()
    float_cols = out.select_dtypes(include=['float32', 'float64']).columns
    out[float_cols] = out[float_cols].round(ndigits)
    return out

In [ ]:
loader = load_data_dynamic_processed(config.mac_data_load_path)
df_map, _, _, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=10,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=True,
)
keys = sorted(df_map)
print(f'Loaded {len(keys)} time slices. monthly_mean={monthly_mean:.4f}')
print(keys[:2], '...', keys[-2:])

first_df = df_map[keys[0]].reset_index(drop=True)
required_cols = {'Latitude', 'Longitude', 'Source_Latitude', 'Source_Longitude', 'ColumnAmountO3'}
missing_cols = required_cols.difference(first_df.columns)
if missing_cols:
    raise ValueError(f'Missing expected columns: {missing_cols}')

# Regular-grid order for Column experiments.  This is deliberately not max-min.
# Sorting by lon then lat makes each contiguous block a vertical grid column.
grid_order = (
    first_df
    .assign(_orig_idx=np.arange(len(first_df)))
    .sort_values(['Longitude', 'Latitude', '_orig_idx'], kind='mergesort')['_orig_idx']
    .to_numpy(dtype=np.int64)
)

grid_coords_full = first_df.iloc[grid_order][['Latitude', 'Longitude']].to_numpy(dtype=np.float64)
print('Full grid:', grid_coords_full.shape)
print('unique lat/lon:', len(np.unique(np.round(grid_coords_full[:, 0], 6))), len(np.unique(np.round(grid_coords_full[:, 1], 6))))

In [ ]:
def load_day_map_regular(day_idx):
    day_keys = keys[day_idx * 8:(day_idx + 1) * 8]
    if len(day_keys) != 8:
        raise ValueError(f'day_idx={day_idx} has {len(day_keys)} slices, expected 8')
    day_df_map = {k: df_map[k] for k in day_keys}
    day_map, _ = loader.load_working_data(
        day_df_map,
        monthly_mean=monthly_mean,
        idx_for_datamap=[0, len(day_keys)],
        ord_mm=grid_order,
        dtype=DTYPE,
        keep_ori=True,
    )
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    return date_str, day_keys, day_map


def make_stride_setup(day_map, stride):
    n_full = grid_coords_full.shape[0]
    thin_idx = np.arange(0, n_full, int(stride), dtype=np.int64)
    thin_map = {k: v[thin_idx].contiguous() for k, v in day_map.items()}
    thin_grid_coords = np.ascontiguousarray(grid_coords_full[thin_idx])
    return {
        'stride': int(stride),
        'thin_idx': thin_idx,
        'input_map': thin_map,
        'grid_coords': thin_grid_coords,
        'n_grid': int(thin_grid_coords.shape[0]),
        'n_valid': count_valid(thin_map),
    }

setups = {}
for day_idx in DAY_IDX_LIST:
    date_str, day_keys, day_map = load_day_map_regular(day_idx)
    print(f'\n{date_str}: {day_keys[0]} ... {day_keys[-1]}')
    for stride in RESOLUTION_STRIDES:
        setup = make_stride_setup(day_map, stride)
        setups[(day_idx, stride)] = setup
        print(
            f"  x{stride}: n_grid={setup['n_grid']:,}  n_valid={setup['n_valid']:,}  "
            f"valid_per_grid_time={setup['n_valid'] / (setup['n_grid'] * 8):.4f}"
        )
    del day_map
    gc.collect()

In [ ]:
def fit_column_resolution(day_idx, stride, smooth):
    setup = setups[(day_idx, stride)]
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    input_map = {k: v.to(DEVICE) for k, v in setup['input_map'].items()}
    grid_coords = setup['grid_coords']

    params = make_params()
    model = ColumnSTTrendVecchiaFit(
        smooth=smooth,
        input_map=input_map,
        grid_coords=grid_coords,
        mm_cond_number=300,
        nheads=0,
        head_right_cols=COLUMN_SPEC['head_right_cols'],
        above_count=COLUMN_SPEC['above_count'],
        right_col_count=COLUMN_SPEC['right_col_count'],
        per_lag_conditioning_count=COLUMN_SPEC['per_lag_conditioning_count'],
        lag_count=COLUMN_SPEC['lag_count'],
        include_lag_self=COLUMN_SPEC['include_lag_self'],
        target_chunk_size=COLUMN_SPEC['target_chunk_size'],
        use_data_coords_for_offsets=True,
        mean_design=MEAN_DESIGN,
    )

    print(f"\n--- fitting {date_str} stride x{stride} smooth={smooth} ---")
    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre

    optimizer = model.set_optimizer(
        params,
        lr=LBFGS_LR,
        max_iter=LBFGS_EVAL,
        max_eval=LBFGS_EVAL,
        history_size=LBFGS_HIST,
    )
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=GRAD_TOL)
    fit_s = time.time() - t_fit

    est = backmap_params(out)
    diag = column_diag(model)
    result = {
        'date_str': date_str,
        'day_idx': day_idx,
        'resolution_stride': int(stride),
        'resolution_label': f'x{stride}',
        'smooth': float(smooth),
        'mean_design': MEAN_DESIGN,
        'model': COLUMN_SPEC['model'],
        'kernel': 'column_st_trend_realloc_stride_grid_thin',
        'coord_mode': 'regular-grid every-k thinning; covariance on Source_Latitude/Source_Longitude',
        'loss': float(out[-1]),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter + 1),
        'precompute_s': float(pre_s),
        'fit_s': float(fit_s),
        'total_s': float(pre_s + fit_s),
        'n_grid': setup['n_grid'],
        'n_valid': setup['n_valid'],
        'valid_per_grid_time': setup['n_valid'] / (setup['n_grid'] * 8),
        **{f'est_{k}': est[k] for k in P_LABELS},
        **diag,
        'head_right_cols': COLUMN_SPEC['head_right_cols'],
        'above_count': COLUMN_SPEC['above_count'],
        'right_col_count': COLUMN_SPEC['right_col_count'],
        'per_lag_conditioning_count': COLUMN_SPEC['per_lag_conditioning_count'],
        'lag_count': COLUMN_SPEC['lag_count'],
        'include_lag_self': COLUMN_SPEC['include_lag_self'],
    }
    print('RESULT:', {k: round(v, 4) if isinstance(v, float) else v for k, v in result.items() if k in ['resolution_label','loss','total_s','n_grid','n_valid','est_sigmasq','est_range_lat','est_range_lon','est_range_time','est_advec_lat','est_advec_lon','est_nugget']})

    del model, optimizer, params
    gc.collect()
    return result

In [ ]:
results = []
for day_idx in DAY_IDX_LIST:
    for smooth in FIT_SMOOTHS:
        for stride in RESOLUTION_STRIDES:
            results.append(fit_column_resolution(day_idx, stride, smooth))
            gc.collect()

results_df = pd.DataFrame(results)
results_df_rounded = round_float_columns(results_df, 4)
results_path = OUT_DIR / f'{OUT_PREFIX}_results.csv'
results_df_rounded.to_csv(results_path, index=False, float_format='%.4f')
print(f'\nSaved results: {results_path}')
results_df_rounded

In [ ]:
param_rows = []
for _, row in results_df.iterrows():
    for p in P_LABELS:
        param_rows.append({
            'date_str': row['date_str'],
            'resolution_stride': int(row['resolution_stride']),
            'resolution_label': row['resolution_label'],
            'smooth': row['smooth'],
            'mean_design': row['mean_design'],
            'model': row['model'],
            'parameter': p,
            'estimate': row[f'est_{p}'],
            'loss': row['loss'],
            'n_grid': row['n_grid'],
            'n_valid': row['n_valid'],
            'total_s': row['total_s'],
        })
param_df = pd.DataFrame(param_rows)
param_df_rounded = round_float_columns(param_df, 4)
param_path = OUT_DIR / f'{OUT_PREFIX}_param_table.csv'
param_df_rounded.to_csv(param_path, index=False, float_format='%.4f')
print(f'Saved parameter table: {param_path}')
param_df_rounded

In [ ]:
summary_cols = [
    'date_str', 'resolution_label', 'n_grid', 'n_valid', 'loss', 'total_s',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'est_nugget', 'mean_m', 'max_m', 'largest_batch_n'
]
print('Column resolution trajectory')
display(results_df_rounded[summary_cols])

pivot = param_df_rounded.pivot_table(
    index='parameter',
    columns='resolution_label',
    values='estimate',
    aggfunc='first',
)
pivot = pivot.reindex(index=P_LABELS, columns=[f'x{s}' for s in RESOLUTION_STRIDES])
print('Parameter trajectory by resolution')
display(pivot)

## Parameter trajectory plots

Each parameter is plotted separately across the regular-grid thinning levels.


In [ ]:
import matplotlib.pyplot as plt

plot_df = param_df_rounded.copy()
plot_df['resolution_label'] = pd.Categorical(
    plot_df['resolution_label'],
    categories=[f'x{s}' for s in RESOLUTION_STRIDES],
    ordered=True,
)
plot_df = plot_df.sort_values(['parameter', 'resolution_label'])

params_to_plot = P_LABELS
n_cols = 2
n_rows = int(np.ceil(len(params_to_plot) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3.1 * n_rows), constrained_layout=True)
axes = np.asarray(axes).ravel()

for ax, param in zip(axes, params_to_plot):
    sub = plot_df[plot_df['parameter'] == param].sort_values('resolution_label')
    ax.plot(
        sub['resolution_label'].astype(str),
        sub['estimate'],
        marker='o',
        linewidth=2.0,
        markersize=6,
    )
    ax.set_title(param)
    ax.set_xlabel('resolution thinning')
    ax.set_ylabel('estimate')
    ax.grid(True, alpha=0.3)

for ax in axes[len(params_to_plot):]:
    ax.axis('off')

fig.suptitle('Column Vecchia parameter estimates by regular-grid thinning', fontsize=14)
plot_path = OUT_DIR / f'{OUT_PREFIX}_parameter_trajectories.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
print(f'Saved plot: {plot_path}')
plt.show()